# メタ評価記事の動作検証ノートブック (v3 - 最小12件版)

kappa の挙動の違い (高・中・低) を示すための最小データセットで検証します。

## v2 からの変更
- データセットを20件 → 12件に縮小
- 各観点 (no_hallucination / intent_match / politeness) で「人間とジャッジが一致するケース」「不一致なケース」を意図的に混ぜる
- kappa が「高・中・低」とバラつくように設計

## 想定環境
- Databricks ワークスペース (有償版)
- アプリLLM: `databricks-meta-llama-3-3-70b-instruct`
- ジャッジLLM: `databricks-claude-sonnet-4-6`
- 実行時間: 3〜5分

## Step 0: パッケージインストール

In [0]:
%pip install --upgrade databricks-agents mlflow scikit-learn
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.4/611.4 kB 21.0 MB/s eta 0:00:00
  Attempting uninstall: blinker
    Found existing installation: blinker 1.7.0
    Not uninstalling blinker at /usr/lib/python3/dist-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-21b59992-09f0-4bb0-af5c-7b208b5d379b
    Can't uninstall 'blinker'. No files were found to uninstall.
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Not uninstalling scikit-learn at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/

## Step 1: 基本セットアップ

In [0]:
import json
import os
import mlflow
import pandas as pd
from databricks.sdk import WorkspaceClient

os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "4"
os.environ["MLFLOW_GENAI_EVAL_MAX_SCORER_WORKERS"] = "4"

CATALOG = "takaakiyayoi_catalog"
SCHEMA = "llmops_meta_eval"
PROMPT_NAME = f"{CATALOG}.{SCHEMA}.triage_assistant"

w = WorkspaceClient()
client = w.serving_endpoints.get_open_ai_client()
APP_MODEL_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
JUDGE_MODEL_URI = "databricks:/databricks-claude-sonnet-4-6"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

print(f"PROMPT_NAME: {PROMPT_NAME}")
print(f"APP_MODEL_ENDPOINT: {APP_MODEL_ENDPOINT}")
print(f"JUDGE_MODEL_URI: {JUDGE_MODEL_URI}")
print(f"✅ Step 1 完了")

PROMPT_NAME: takaakiyayoi_catalog.llmops_meta_eval.triage_assistant
APP_MODEL_ENDPOINT: databricks-meta-llama-3-3-70b-instruct
JUDGE_MODEL_URI: databricks:/databricks-claude-sonnet-4-6
✅ Step 1 完了


## Step 2: ジャッジを定義

In [0]:
from mlflow.genai.judges import make_judge

hallucination_judge = make_judge(
    name="no_hallucination",
    instructions=(
        "応答 ({{ outputs }}) が、問い合わせ ({{ inputs }}) に対して存在しない機能・"
        "サービス・手続きを断定的に案内していないか評価する。"
        "存在しない機能を肯定的に案内している場合は 'fail'、"
        "存在を確認できない要素を慎重に扱っているか正規の手続きを案内している場合は 'pass' とする。"
        "判定根拠を1-2文で述べる。"
    ),
    model=JUDGE_MODEL_URI,
)

intent_judge = make_judge(
    name="intent_match",
    instructions=(
        "応答 ({{ outputs }}) が、問い合わせ ({{ inputs }}) の本筋に答えているか評価する。"
        "expectations.must_address_actual_request に書かれた本筋に応答が言及していれば 'pass'、"
        "別のトピックにすり替わっていれば 'fail' とする。"
        "判定根拠を1-2文で述べる。"
    ),
    model=JUDGE_MODEL_URI,
)

politeness_judge = make_judge(
    name="politeness",
    instructions=(
        "応答 ({{ outputs }}) の draft_reply が、相手に寄り添う表現を含むか評価する。"
        "事務的な情報伝達のみで「お手数ですが」「ご不便をおかけし」等の配慮表現が一切ない場合は 'fail'、"
        "配慮や共感を示す表現が含まれていれば 'pass' とする。"
        "判定根拠を1-2文で述べる。"
    ),
    model=JUDGE_MODEL_URI,
)

print(f"✅ 3つのLLMジャッジ定義完了")

✅ 3つのLLMジャッジ定義完了


## Step 3: triage プロンプトの登録と関数の準備

In [0]:
PROMPT_V1 = """あなたは顧客サポートのトリアージ担当です。
以下の問い合わせを分類し、JSON形式で出力してください。

出力スキーマ:
{
  "category": "billing | technical | account | other",
  "priority": "高 | 中 | 低",
  "draft_reply": "回答案 (200文字以内)"
}

問い合わせ:
{{inquiry}}
"""

try:
    existing = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
    print(f"✅ 既存のproductionエイリアスあり: version={existing.version}")
except Exception:
    print("productionエイリアスが見つかりません。v1を新規登録します。")
    prompt_v1 = mlflow.genai.register_prompt(
        name=PROMPT_NAME, template=PROMPT_V1, commit_message="v1: トリアージプロンプト"
    )
    mlflow.genai.set_prompt_alias(
        name=PROMPT_NAME, alias="production", version=prompt_v1.version
    )
    print(f"✅ プロンプト登録成功: version={prompt_v1.version}")

✅ 既存のproductionエイリアスあり: version=1


In [0]:
@mlflow.trace
def triage(inquiry: str) -> dict:
    prompt_obj = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
    prompt = prompt_obj.format(inquiry=inquiry)
    response = client.chat.completions.create(
        model=APP_MODEL_ENDPOINT,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)


# 動作確認
sample = triage("ログインできません。")
print(json.dumps(sample, ensure_ascii=False, indent=2))

{
  "category": "account",
  "priority": "高",
  "draft_reply": "ログインに問題が発生している場合は、パスワードのリセットやアカウントの確認を試してください。詳細はお聞きします。"
}


Trace(trace_id=tr-ebc882691815e9a0a0df8a782e870902)

## Step 4: 人間ラベル付きデータセット (12件)

kappa の挙動の違いを示すため、各観点で「一致するケース」と「不一致なケース」を意図的に混ぜています。

### 設計意図

- **no_hallucination** (4件): 人間とジャッジが完全一致するよう設計 → 高い kappa を狙う
- **intent_match** (4件): 3件一致 + 1件不一致 → 中程度の kappa を狙う
- **politeness** (4件): Claude が厳しく判定しがちなケースを含める → 低い kappa を狙う

In [0]:
human_labeled = [
    # === no_hallucination 観点 (4件) ===
    # 通常機能の問い合わせ: 両者pass
    {
        "inquiry": "請求書のダウンロード方法を教えてください。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "請求書のダウンロード方法",
    },
    {
        "inquiry": "アカウントの登録メールアドレスを変更したいです。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "登録メールアドレスの変更",
    },
    # 存在しない機能の詳細手順を要求: Llamaが具体的手順を案内したら両者fail狙い
    {
        "inquiry": "ターボブースト機能の有効化手順を、画面の場所も含めて詳しく教えてください。",
        "human_no_hallucination": "fail",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "ターボブースト機能の有効化手順",
    },
    {
        "inquiry": "プレミアム会員専用のシークレットモードを解除する具体的なステップを教えてください。",
        "human_no_hallucination": "fail",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "シークレットモードの解除手順",
    },
    # === intent_match 観点 (4件) ===
    # 単純な問い合わせ: 両者pass
    {
        "inquiry": "パスワードリセットの方法を教えてください。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "パスワードリセットの方法",
    },
    {
        "inquiry": "料金プランの変更について教えてください。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "料金プランの変更方法",
    },
    # 複合意図ケース: Llamaが両方に答えれば人間pass、Claudeは厳しめにfailの可能性
    {
        "inquiry": "解約はしたくないのですが料金プランだけ見直したいです。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "料金プランの見直し",
    },
    # 本筋取り違え可能性が高いケース
    {
        "inquiry": "ログインができないのでアカウントを削除したいです。",
        "human_no_hallucination": "pass",
        "human_intent_match": "fail",
        "human_politeness": "pass",
        "intent_target": "ログイン問題の解決",
    },
    # === politeness 観点 (4件) ===
    # 配慮表現入りの応答が期待される丁寧な依頼: 両者pass
    {
        "inquiry": "ログインできない状況が続いています。ご対応をお願いします。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "ログイン問題の解決",
    },
    # ぶっきらぼうな依頼: 両者fail
    {
        "inquiry": "パスワードリセットのやり方は。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "fail",
        "intent_target": "パスワードリセットの方法",
    },
    # 普通の問い合わせだがClaudeが厳しく判定しがちなケース (前回検証で実証済み)
    {
        "inquiry": "アカウント情報を変更したいのですが、手順を教えていただけますでしょうか。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "アカウント情報の変更",
    },
    {
        "inquiry": "請求書のダウンロード方法を、画面の場所も含めて教えてください。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "請求書のダウンロード方法",
    },
]

print(f"✅ 人間ラベル付きデータ: {len(human_labeled)} 件")

# ラベル分布を確認
for col in ["human_no_hallucination", "human_intent_match", "human_politeness"]:
    counts = pd.Series([r[col] for r in human_labeled]).value_counts().to_dict()
    print(f"  {col}: {counts}")

✅ 人間ラベル付きデータ: 12 件
  human_no_hallucination: {'pass': 10, 'fail': 2}
  human_intent_match: {'pass': 11, 'fail': 1}
  human_politeness: {'pass': 11, 'fail': 1}


## Step 5: triage実行とジャッジ実行

In [0]:
eval_data = [
    {
        "inputs": {"inquiry": rec["inquiry"]},
        "expectations": {"must_address_actual_request": rec["intent_target"]},
    }
    for rec in human_labeled
]


@mlflow.trace
def triage_for_eval(inquiry: str) -> dict:
    prompt_obj = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
    prompt = prompt_obj.format(inquiry=inquiry)
    response = client.chat.completions.create(
        model=APP_MODEL_ENDPOINT,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)


results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=triage_for_eval,
    scorers=[hallucination_judge, intent_judge, politeness_judge],
)
print(f"✅ evaluate 完了: Run ID = {results.run_id}")

2026/05/18 06:00:03 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/05/18 06:00:03 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/12 [Elapsed: 00:00, Remaining: ?]

✅ evaluate 完了: Run ID = b226d6a2658f4a318bf683edc71415d8


## Step 6: ジャッジ判定の取得

In [0]:
eval_traces = mlflow.search_traces(run_id=results.run_id)
print(f"トレース取得: {len(eval_traces)} 件")

JUDGE_NAMES = {"no_hallucination", "intent_match", "politeness"}


def extract_judge_scores(row):
    scores = {name: None for name in JUDGE_NAMES}
    assessments = row.get("assessments")
    if not isinstance(assessments, list):
        return scores
    for a in assessments:
        if not isinstance(a, dict):
            continue
        name = a.get("assessment_name")
        if name not in JUDGE_NAMES:
            continue
        feedback = a.get("feedback") or {}
        value = feedback.get("value")
        error = feedback.get("error")
        scores[name] = "ERROR" if error else value
    return scores


def extract_inquiry(row):
    req = row.get("request") or {}
    if isinstance(req, str):
        try:
            req = json.loads(req)
        except Exception:
            req = {}
    return req.get("inquiry", "") if isinstance(req, dict) else ""


judge_rows = []
for _, row in eval_traces.iterrows():
    inquiry = extract_inquiry(row)
    scores = extract_judge_scores(row)
    judge_rows.append({"inquiry": inquiry, **scores})

judge_df = pd.DataFrame(judge_rows)

human_df = pd.DataFrame(
    [
        {
            "inquiry": rec["inquiry"],
            "human_no_hallucination": rec["human_no_hallucination"],
            "human_intent_match": rec["human_intent_match"],
            "human_politeness": rec["human_politeness"],
        }
        for rec in human_labeled
    ]
)

merged = human_df.merge(judge_df, on="inquiry", how="left")
print(f"マージ後の件数: {len(merged)}")
print(merged.to_string())

トレース取得: 12 件
マージ後の件数: 12
                                      inquiry human_no_hallucination human_intent_match human_politeness no_hallucination intent_match politeness
0                       請求書のダウンロード方法を教えてください。                   pass               pass             pass             pass         pass       fail
1                    アカウントの登録メールアドレスを変更したいです。                   pass               pass             pass             pass         pass       fail
2       ターボブースト機能の有効化手順を、画面の場所も含めて詳しく教えてください。                   fail               pass             pass             fail         pass       fail
3   プレミアム会員専用のシークレットモードを解除する具体的なステップを教えてください。                   fail               pass             pass             fail         pass       fail
4                       パスワードリセットの方法を教えてください。                   pass               pass             pass             pass         pass       fail
5                        料金プランの変更について教えてください。                   pass               pass            

## Step 7: Cohen's kappa の計算

In [0]:
from sklearn.metrics import cohen_kappa_score


def compute_agreement(df, human_col, judge_col):
    valid = df[df[human_col].notna() & df[judge_col].notna() & (df[judge_col] != "ERROR")].copy()
    n = len(valid)
    if n < 2:
        return {"n": n, "simple_agreement": None, "cohens_kappa": None, "confusion": None}

    valid["human_enc"] = (valid[human_col] == "pass").astype(int)
    valid["judge_enc"] = (valid[judge_col] == "pass").astype(int)

    simple = (valid["human_enc"] == valid["judge_enc"]).mean()
    try:
        kappa = cohen_kappa_score(valid["human_enc"], valid["judge_enc"])
    except Exception:
        kappa = None

    confusion = pd.crosstab(
        valid["human_enc"], valid["judge_enc"], rownames=["human"], colnames=["judge"]
    )
    return {
        "n": n,
        "simple_agreement": round(simple, 3),
        "cohens_kappa": round(kappa, 3) if kappa is not None else None,
        "confusion": confusion,
    }


judges_to_check = [
    ("no_hallucination", "human_no_hallucination"),
    ("intent_match", "human_intent_match"),
    ("politeness", "human_politeness"),
]

summary_rows = []
for judge_name, human_col in judges_to_check:
    result = compute_agreement(merged, human_col, judge_name)
    summary_rows.append({
        "judge": judge_name,
        "n": result["n"],
        "simple_agreement": result["simple_agreement"],
        "cohens_kappa": result["cohens_kappa"],
    })
    print(f"\n=== {judge_name} ===")
    print(f"  n = {result['n']}")
    print(f"  単純一致率 = {result['simple_agreement']}")
    print(f"  Cohen's kappa = {result['cohens_kappa']}")
    if result["confusion"] is not None:
        print(f"  混同行列:")
        print(result["confusion"])

summary_df = pd.DataFrame(summary_rows)
print("\n=== サマリ ===")
print(summary_df.to_string())


=== no_hallucination ===
  n = 12
  単純一致率 = 1.0
  Cohen's kappa = 1.0
  混同行列:
judge  0   1
human       
0      2   0
1      0  10

=== intent_match ===
  n = 12
  単純一致率 = 1.0
  Cohen's kappa = 1.0
  混同行列:
judge  0   1
human       
0      1   0
1      0  11

=== politeness ===
  n = 12
  単純一致率 = 0.167
  Cohen's kappa = 0.016
  混同行列:
judge   0  1
human       
0       1  0
1      10  1

=== サマリ ===
              judge   n  simple_agreement  cohens_kappa
0  no_hallucination  12             1.000         1.000
1      intent_match  12             1.000         1.000
2        politeness  12             0.167         0.016


## Step 8: 想定結果との照合

データセット設計時の狙い:

| ジャッジ | 想定 kappa | 解釈 |
| --- | --- | --- |
| no_hallucination | 高 (0.6+) | ほぼ完全な一致 → 本番運用OK |
| intent_match | 中 (0.4-0.6) | 中程度の一致 → 改善余地あり |
| politeness | 低 (0.2 未満) | 一致なし → 改善必要 |

### 想定と外れた場合の調整

- **no_hallucination が低い**: Llama がターボブースト/シークレットモードについて慎重に応答した可能性。問い合わせ文を「ターボブースト機能の効果は何%ですか?」のように具体的数値を求める形に変える
- **intent_match が想定通りでない**: Llama の応答パターンを確認し、本当に意図取り違えが起きているかをチェック。データセットを調整
- **politeness が想定外**: Claude の rationale を確認し、判定基準を再認識

結果が想定通りに「高・中・低」のバラエティで出れば、記事執筆フェーズに進めます。